# Memory AI Lab — Évaluation ARI V3

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

**Données requises sur Google Drive :**
```
Mon Drive/memory_ai_data/
  group_anon.txt
  group_gold.json
```

In [ ]:
# ── CELLULE 1 : Cloner le code depuis GitHub ───────────────────────────────
import os

REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'

if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull
else:
    !git clone {REPO} {CODE_DIR}

import sys
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Installer les dépendances ─────────────────────────────────
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!python -m spacy download fr_core_news_sm -q
print('✓ Dépendances OK')

In [ ]:
# ── CELLULE 3 : Monter Drive pour les données ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/memory_ai_data'

import os
assert os.path.exists(f'{DATA_DIR}/group_anon.txt'), \
    f'Fichier manquant — upload group_anon.txt dans {DATA_DIR}/'
assert os.path.exists(f'{DATA_DIR}/group_gold.json'), \
    f'Fichier manquant — upload group_gold.json dans {DATA_DIR}/'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ Drive monté | Device : {device}')
if device == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── CELLULE 4 : Parse + Embeddings (GPU + cache Drive) ────────────────────
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings.npy'

print('[1/3] Parsing ...')
artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [a.content for a in artifacts]
print(f'      {len(texts)} messages')

if EMBED_CACHE.exists():
    print('[2/3] Embeddings (cache Drive) ...')
    embeddings = np.load(EMBED_CACHE)
    assert len(embeddings) == len(texts), 'Cache périmé — supprimer group_embeddings.npy'
else:
    print(f'[2/3] Embeddings sur {device} ...')
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
    embeddings = model.encode(
        texts, batch_size=256, show_progress_bar=True,
        device=device, convert_to_numpy=True
    ).astype(np.float32)
    np.save(EMBED_CACHE, embeddings)
    print(f'      Sauvegardé → {EMBED_CACHE}')

print(f'[3/3] Shape embeddings : {embeddings.shape}')

In [ ]:
# ── CELLULE 5 : Gold labels ────────────────────────────────────────────────
import json

with open(f'{DATA_DIR}/group_gold.json', encoding='utf-8') as f:
    gold = json.load(f)

n = len(artifacts)
y_true = [None] * n
for ep in gold['episodes']:
    for idx in range(ep['start_idx'], ep['end_idx'] + 1):
        if idx < n:
            y_true[idx] = ep['episode_id']

n_ep_gold = len(set(l for l in y_true if l is not None))
print(f'✓ {n_ep_gold} épisodes gold sur {sum(1 for l in y_true if l is not None)} messages')

In [ ]:
# ── CELLULE 6 : Segmentation V3 — paramètres modifiables ──────────────────
from episode_algorithm import EpisodeSegmenter
from episode_splitter import EpisodeSplitter, SplitConfig
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

SEG_KWARGS = dict(
    time_threshold_minutes=120,
    attach_threshold=0.30,
    alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
    dormancy_minutes=1440,
    ema_alpha=0.80,
    active_penalty_hours=24.0,
    hard_break_minutes=720,
    allow_reactivation=True,
)

segmenter = EpisodeSegmenter(**SEG_KWARGS)
episodes  = segmenter.segment(artifacts, embeddings)
episodes  = segmenter.consolidate(episodes)
episodes  = EpisodeSplitter(SplitConfig(
    min_cohesion=0.65, min_size_to_split=8,
    max_span_hours=168.0, max_splits=6,
    min_sub_size=3, silhouette_threshold=0.10,
)).split(episodes, artifacts, embeddings)

y_pred = [None] * n
for ep in episodes:
    for idx in ep.artifact_indices:
        if idx < n: y_pred[idx] = ep.id

pairs = [(t, p) for t, p in zip(y_true, y_pred) if t is not None and p is not None]
yt, yp = zip(*pairs)
ari = adjusted_rand_score(yt, yp)
nmi = normalized_mutual_info_score(yt, yp)

print(f"""
╔══════════════════════════════════╗
║  ARI  : {ari:+.4f}               ║
║  NMI  : {nmi:.4f}                ║
║  Gold : {n_ep_gold} épisodes     ║
║  Pred : {len(episodes)} épisodes ║
╚══════════════════════════════════╝
""")

In [ ]:
# ── CELLULE 7 (optionnelle) : Grid search ─────────────────────────────────
# Décommenter pour chercher les meilleurs paramètres

# import itertools, pandas as pd
# results = []
# for attach, hb in itertools.product([0.20, 0.25, 0.30, 0.35], [360, 720, 1440, 0]):
#     seg = EpisodeSegmenter(time_threshold_minutes=120, attach_threshold=attach,
#         alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
#         dormancy_minutes=1440, ema_alpha=0.80, active_penalty_hours=24.0,
#         hard_break_minutes=hb, allow_reactivation=True)
#     eps = seg.consolidate(seg.segment(artifacts, embeddings))
#     yp = [None] * n
#     for ep in eps:
#         for idx in ep.artifact_indices:
#             if idx < n: yp[idx] = ep.id
#     pairs2 = [(t,p) for t,p in zip(y_true, yp) if t and p]
#     a = adjusted_rand_score(*zip(*pairs2))
#     results.append({'attach': attach, 'hard_break': hb, 'n_ep': len(eps), 'ari': a})
#     print(f'attach={attach}  hb={hb:5d}  ARI={a:.4f}  ep={len(eps)}')
# pd.DataFrame(results).sort_values('ari', ascending=False)